In [0]:

from pyspark.sql import functions as F
from datetime import datetime

# ==========================================
# E-COMMERCE DATA ENGINEERING PROJECT
# BRONZE LAYER - RAW DATA INGESTION
# ==========================================

SOURCE_PATH = "/Volumes/workspace/ecommerce/raw_files/"
TARGET_SCHEMA = "workspace.ecommerce"

# Create target schema if necessary
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce")

# All nine source datasets
datasets = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

results = []

for table_name, file_name in datasets.items():

    print(f"Processing: {file_name}")

    file_path = SOURCE_PATH + file_name

    # Read original CSV without modifying its contents
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .csv(file_path)
    )

    # Add ingestion metadata
    df = (
        df.withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.lit(file_name))
    )

    # Write as a managed Delta table
    target_table = f"{TARGET_SCHEMA}.bronze_{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    row_count = spark.table(target_table).count()

    results.append((table_name, row_count, target_table))

    print(f"SUCCESS: {target_table} | Rows: {row_count}")

# Display ingestion summary
summary_df = spark.createDataFrame(
    results,
    ["dataset", "row_count", "table_name"]
)

print("==========================================")
print("BRONZE INGESTION COMPLETED")
print("==========================================")

display(summary_df)


Processing: olist_customers_dataset.csv
SUCCESS: workspace.ecommerce.bronze_customers | Rows: 99441
Processing: olist_geolocation_dataset.csv
SUCCESS: workspace.ecommerce.bronze_geolocation | Rows: 1000163
Processing: olist_order_items_dataset.csv
SUCCESS: workspace.ecommerce.bronze_order_items | Rows: 112650
Processing: olist_order_payments_dataset.csv
SUCCESS: workspace.ecommerce.bronze_order_payments | Rows: 103886
Processing: olist_order_reviews_dataset.csv
SUCCESS: workspace.ecommerce.bronze_order_reviews | Rows: 104162
Processing: olist_orders_dataset.csv
SUCCESS: workspace.ecommerce.bronze_orders | Rows: 99441
Processing: olist_products_dataset.csv
SUCCESS: workspace.ecommerce.bronze_products | Rows: 32951
Processing: olist_sellers_dataset.csv
SUCCESS: workspace.ecommerce.bronze_sellers | Rows: 3095
Processing: product_category_name_translation.csv
SUCCESS: workspace.ecommerce.bronze_category_translation | Rows: 71
BRONZE INGESTION COMPLETED


dataset,row_count,table_name
customers,99441,workspace.ecommerce.bronze_customers
geolocation,1000163,workspace.ecommerce.bronze_geolocation
order_items,112650,workspace.ecommerce.bronze_order_items
order_payments,103886,workspace.ecommerce.bronze_order_payments
order_reviews,104162,workspace.ecommerce.bronze_order_reviews
orders,99441,workspace.ecommerce.bronze_orders
products,32951,workspace.ecommerce.bronze_products
sellers,3095,workspace.ecommerce.bronze_sellers
category_translation,71,workspace.ecommerce.bronze_category_translation
